# 생성형 AI 기반 영농일지 자동 작성 실습 답안

농가의 **구어체 메모**를 02번 SQLite 창고(기상·농가·생육·병해충) 근거로
**구조화된 영농일지**로 바꾸고 `farm_diary` 테이블에 저장합니다.

```text
사용자: "어제 고추밭 일 한 거 일지로 남겨줘."
                │
                ▼
           create_agent
  메모 해석 / 당일 기상 / 농가·생육 / 병해충 검색
                │
                ▼
         FarmDiary JSON 검증
                │
                ▼
     C:\env\farm_warehouse\farm_warehouse.db  (farm_diary)
```

- OpenAI 키: `C:\env\.env` 의 `OPENAI_API_KEY`
- 기상 근거: 2023년 전북 완주군 (반교리 우선)
- 농가·생육: 2024년 참고. 작업일 기상으로 쓰지 않음
- 메모에 없는 약제명·수확량은 만들지 않음


## STEP 0. 환경 준비


In [1]:
import json
import os

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from langchain.agents import create_agent

from farm_diary_core import (
    SAMPLE_MEMOS,
    SAMPLE_QUESTIONS,
    TOOL_LABELS,
    FarmDiary,
    ask_agent,
    build_runtime,
    ensure_warehouse,
    load_diary_table,
    reset_diary_table,
    sample_weather_rows,
    warehouse_overview,
)

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_colwidth", 80)

load_dotenv(r"C:\env\.env")
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(r"OPENAI_API_KEY 를 C:\env\.env 에서 찾을 수 없습니다.")
print("OPENAI_API_KEY 로드 완료 (키 값은 출력하지 않습니다)")
print("create_agent:", create_agent.__module__)
print("LLM: gpt-4o-mini")


OPENAI_API_KEY 로드 완료 (키 값은 출력하지 않습니다)
create_agent: langchain.agents.factory
LLM: gpt-4o-mini


## STEP 1. 창고 연결 확인

02번에서 적재한 `C:\env\farm_warehouse\farm_warehouse.db` 를 읽습니다.
비어 있으면 02 폴더 `data/*.csv` 를 같은 스키마로 적재합니다.
포털을 다시 받지 않습니다.


In [2]:
print("\n".join(ensure_warehouse()))
display(warehouse_overview())

print("시나리오용 기상 샘플 (2023-08-10 / 15 / 18)")
weather_sample = sample_weather_rows()
display(weather_sample)

bangyo = weather_sample[weather_sample["station"].astype(str).str.contains("반교리")]
for _, row in bangyo.iterrows():
    rain = row["rainfall_mm"]
    note = ""
    if rain >= 400:
        note = " ← 이상치(일지 강수량으로 쓰지 않음)"
    elif rain >= 20:
        note = " ← 폭우"
    print(
        f"{row['obs_date']} {row['station']}: "
        f"최저 {row['tmin_c']} / 최고 {row['tmax_c']} / 습도 {row['humidity_pct']} / 강수 {rain}mm{note}"
    )
print("이서면 2023-08-10 강수가 400mm를 넘으면 센서/원자료 이상으로만 참고합니다.")


창고 테이블 weather/farm/growth/pest_info 가 이미 있습니다.


,table,건수,기간,설명
0,weather,730,2023-01-01 ~ 2023-12-31,2023 완주 기상
1,farm,312,2023-09-10 ~ 2024-10-17,2024 노지 농가
2,growth,10914,2024-01-10 ~ 2024-11-08,2024 노지 생육
3,pest_info,999,-,병해충 목록


시나리오용 기상 샘플 (2023-08-10 / 15 / 18)


,station,obs_date,tmin_c,tmax_c,tavg_c,humidity_pct,rainfall_mm
0,완주군 반교리,2023-08-10,24.00,25.00,24.67,93.12,173.0
1,완주군 이서면,2023-08-10,21.92,24.60,23.07,92.29,2458.5
2,완주군 반교리,2023-08-15,24.00,32.00,27.12,85.83,0.0
3,완주군 이서면,2023-08-15,24.42,32.27,27.76,81.21,6.0
4,완주군 반교리,2023-08-18,24.00,32.00,27.33,86.04,20.0
5,완주군 이서면,2023-08-18,23.92,32.33,27.77,77.89,10.5


2023-08-10 완주군 반교리: 최저 24.0 / 최고 25.0 / 습도 93.12 / 강수 173.0mm ← 폭우
2023-08-15 완주군 반교리: 최저 24.0 / 최고 32.0 / 습도 85.83 / 강수 0.0mm
2023-08-18 완주군 반교리: 최저 24.0 / 최고 32.0 / 습도 86.04 / 강수 20.0mm ← 폭우
이서면 2023-08-10 강수가 400mm를 넘으면 센서/원자료 이상으로만 참고합니다.


## STEP 2. 영농일지 스키마 (Pydantic)

저장·검증은 `FarmDiary` 모델만 사용합니다. 자유 텍스트만 저장하지 않습니다.


In [3]:
schema = FarmDiary.model_json_schema()
print(json.dumps(schema, ensure_ascii=False, indent=2)[:2500])
print("...\n필드:", list(FarmDiary.model_fields))


{
  "$defs": {
    "Evidence": {
      "properties": {
        "weather_rows": {
          "default": false,
          "title": "Weather Rows",
          "type": "boolean"
        },
        "farm_rows": {
          "default": false,
          "title": "Farm Rows",
          "type": "boolean"
        },
        "growth_rows": {
          "default": false,
          "title": "Growth Rows",
          "type": "boolean"
        },
        "pest_names": {
          "items": {
            "type": "string"
          },
          "title": "Pest Names",
          "type": "array"
        }
      },
      "title": "Evidence",
      "type": "object"
    },
    "WeatherBlock": {
      "properties": {
        "tmin_c": {
          "anyOf": [
            {
              "type": "number"
            },
            {
              "type": "null"
            }
          ],
          "default": null,
          "title": "Tmin C"
        },
        "tmax_c": {
          "anyOf": [
            {
           

## STEP 3. 샘플 메모 (코드 상수)


In [4]:
memo_df = pd.DataFrame(SAMPLE_MEMOS)
display(memo_df)
for item in SAMPLE_MEMOS:
    print(f"[{item['id']}] {item['title']}")
    print(item["memo"])
    print()


,id,title,memo
0,A,"관수·시비, 병해 없음",2023년 8월 15일 완주 고추밭. 낮에 더워서 물 한번 줬고 웃거름도 조금 했어. 열매는 달렸는데 갯수는 세지 못함.
1,B,"방제 + 증상만, 약제명 없음",8월 18일 완주 고추. 어제부터 비 오고 습해서 잎에 갈색 반점이 보임. 오후에 약 뿌렸음. 약 이름은 기억 안 남.
2,C,폭우 고랑 정리,2023-08-10 완주. 비가 너무 많이 와서 고추밭 물 빠지나 보고 고랑 정리만 했음. 수확은 안 함.
3,D,완주 양파 잔재 정리,2023년 8월 15일 전북 완주 양파밭은 이미 수확 끝난 뒤라 잔재 정리만 했어.
4,E,토마토(자료 없음),오늘 토마토 하우스 정식하고 점적관수 시작했어. 일지에 품종이랑 오늘 기온 넣어서 작성해줘.


[A] 관수·시비, 병해 없음
2023년 8월 15일 완주 고추밭. 낮에 더워서 물 한번 줬고 웃거름도 조금 했어. 열매는 달렸는데 갯수는 세지 못함.

[B] 방제 + 증상만, 약제명 없음
8월 18일 완주 고추. 어제부터 비 오고 습해서 잎에 갈색 반점이 보임. 오후에 약 뿌렸음. 약 이름은 기억 안 남.

[C] 폭우 고랑 정리
2023-08-10 완주. 비가 너무 많이 와서 고추밭 물 빠지나 보고 고랑 정리만 했음. 수확은 안 함.

[D] 완주 양파 잔재 정리
2023년 8월 15일 전북 완주 양파밭은 이미 수확 끝난 뒤라 잔재 정리만 했어.

[E] 토마토(자료 없음)
오늘 토마토 하우스 정식하고 점적관수 시작했어. 일지에 품종이랑 오늘 기온 넣어서 작성해줘.



## STEP 4~5. Tool 6개 + create_agent

`from langchain.agents import create_agent`


In [5]:
print(reset_diary_table())
runtime = build_runtime()
print("창고 DB:", runtime["db_path"])
print("02 CSV:", runtime["data_dir"])
print("tools:", runtime["tools"])
print("TOOL_LABELS:", TOOL_LABELS)


farm_diary 테이블을 새로 만들었습니다.
창고 DB: C:\env\farm_warehouse\farm_warehouse.db
02 CSV: C:\MyCursorLab\06_농업 업무 자동화 서비스 개발 실습\02_농업 데이터 수집 자동화 시스템 개발 실습\data
tools: ['parse_work_memo', 'get_diary_weather', 'get_crop_field_context', 'search_pest_info', 'save_farm_diary', 'query_saved_diaries']
TOOL_LABELS: {'parse_work_memo': '메모 해석', 'get_diary_weather': '당일 기상', 'get_crop_field_context': '농가·생육', 'search_pest_info': '병해충 검색', 'save_farm_diary': '일지 저장', 'query_saved_diaries': '일지 목록'}


In [6]:
def show_run(item):
    print("=" * 72)
    print(item["label"])
    print("Q:", item["text"])
    out = ask_agent(runtime, item["text"])
    print("호출 Tool:", " → ".join(out["tools"]) if out["tools"] else "(없음)")
    for tr in out["trace"]:
        if tr["label"] == "결과":
            print(f"  [결과:{tr['name']}] {tr.get('preview', '')[:200]}")
        else:
            args = tr.get("args") or {}
            short = {k: (str(v)[:80] + "…" if len(str(v)) > 80 else v) for k, v in args.items()}
            print(f"  → {tr['label']} {short}")
    print("\n[최종 답]\n")
    print(out["answer"])
    return out


## STEP 6. 실행 시나리오 A~E


In [7]:
out_a = show_run(SAMPLE_QUESTIONS[0])


A. 관수·시비
Q: 다음 메모를 영농일지로 작성하고 저장해줘.
2023년 8월 15일 완주 고추밭. 낮에 더워서 물 한번 줬고 웃거름도 조금 했어. 열매는 달렸는데 갯수는 세지 못함.
호출 Tool: parse_work_memo → get_diary_weather → get_crop_field_context → save_farm_diary
  → 메모 해석 {'memo': '2023년 8월 15일 완주 고추밭. 낮에 더워서 물 한번 줬고 웃거름도 조금 했어. 열매는 달렸는데 갯수는 세지 못함.'}
  [결과:parse_work_memo] {"work_date": "2023-08-15", "region": "완주", "crop": "고추", "variety": "", "work_items": [{"category": "관수", "description": "관수", "material": "", "amount": ""}, {"category": "시비", "description": "웃거름", 
  → 당일 기상 {'work_date': '2023-08-15'}
  [결과:get_diary_weather] {"found": true, "work_date": "2023-08-15", "primary": {"station": "완주군 반교리", "tmin_c": 24.0, "tmax_c": 32.0, "tavg_c": 27.12, "humidity_pct": 85.83, "rainfall_mm": 0.0, "source_year": "2023", "flags":
  → 농가·생육 {'crop': '고추'}
  [결과:get_crop_field_context] {"crop": "고추", "in_warehouse": true, "year_note": "farm/growth 는 2024년 참고 정보", "growth_filter_note": "고추 생육은 전국 자료입니다. 전북 필터를 적용하지 않았습니다.", "farm_rows": 8, "farm_sample": [{"s

In [8]:
out_b = show_run(SAMPLE_QUESTIONS[1])


B. 방제(약제명 없음)
Q: 다음 메모를 영농일지로 작성하고 저장해줘.
8월 18일 완주 고추. 어제부터 비 오고 습해서 잎에 갈색 반점이 보임. 오후에 약 뿌렸음. 약 이름은 기억 안 남.
호출 Tool: parse_work_memo → get_diary_weather → search_pest_info → get_crop_field_context → save_farm_diary
  → 메모 해석 {'memo': '8월 18일 완주 고추. 어제부터 비 오고 습해서 잎에 갈색 반점이 보임. 오후에 약 뿌렸음. 약 이름은 기억 안 남.'}
  [결과:parse_work_memo] {"work_date": "2023-08-18", "region": "완주", "crop": "고추", "variety": "", "work_items": [{"category": "방제", "description": "방제", "material": "약제명 미기재", "amount": ""}], "symptoms": "잎에 갈색 반점", "growth_o
  → 당일 기상 {'work_date': '2023-08-18'}
  [결과:get_diary_weather] {"found": true, "work_date": "2023-08-18", "primary": {"station": "완주군 반교리", "tmin_c": 24.0, "tmax_c": 32.0, "tavg_c": 27.33, "humidity_pct": 86.04, "rainfall_mm": 20.0, "source_year": "2023", "flags"
  → 병해충 검색 {'crop': '고추', 'symptom': '잎에 갈색 반점'}
  [결과:search_pest_info] {"crop": "고추", "symptom": "잎에 갈색 반점", "hits": [{"pest_name": "갈색점무늬병", "detail": "etc: , divName: 병, korName: 갈색점무늬병, oprName: Cercospo

In [9]:
out_c = show_run(SAMPLE_QUESTIONS[2])


C. 폭우 고랑 정리
Q: 2023-08-10 완주. 비가 너무 많이 와서 고추밭 물 빠지나 보고 고랑 정리만 했음. 수확은 안 함. 일지로 남겨줘.
호출 Tool: parse_work_memo → get_diary_weather → get_crop_field_context → save_farm_diary
  → 메모 해석 {'memo': '2023-08-10 완주. 비가 너무 많이 와서 고추밭 물 빠지나 보고 고랑 정리만 했음. 수확은 안 함.'}
  [결과:parse_work_memo] {"work_date": "2023-08-10", "region": "완주", "crop": "고추", "variety": "", "work_items": [{"category": "기타", "description": "고랑/잔재 정리", "material": "", "amount": ""}], "symptoms": "", "growth_observatio
  → 당일 기상 {'work_date': '2023-08-10'}
  [결과:get_diary_weather] {"found": true, "work_date": "2023-08-10", "primary": {"station": "완주군 반교리", "tmin_c": 24.0, "tmax_c": 25.0, "tavg_c": 24.67, "humidity_pct": 93.12, "rainfall_mm": 173.0, "source_year": "2023", "flags
  → 농가·생육 {'crop': '고추'}
  [결과:get_crop_field_context] {"crop": "고추", "in_warehouse": true, "year_note": "farm/growth 는 2024년 참고 정보", "growth_filter_note": "고추 생육은 전국 자료입니다. 전북 필터를 적용하지 않았습니다.", "farm_rows": 8, "farm_sample": [{"sido": "경기", "sigungu": "안
  

In [10]:
out_d = show_run(SAMPLE_QUESTIONS[3])


D. 저장 목록
Q: 저장한 영농일지 중에서 고추, 2023년 8월 것만 목록으로 보여줘.
호출 Tool: query_saved_diaries
  → 일지 목록 {'crop': '고추', 'start_date': '2023-08-01', 'end_date': '2023-08-31'}
  [결과:query_saved_diaries] {"count": 3, "items": [{"diary_id": 3, "work_date": "2023-08-10", "crop": "고추", "region": "전북 완주군", "work_summary": "기타:고랑/잔재 정리", "saved_at": "2026-08-31 23:52:02"}, {"diary_id": 1, "work_date": "202

[최종 답]

2023년 8월의 고추 영농일지 목록은 다음과 같습니다:

1. **작업일**: 2023-08-10
   - **작물**: 고추
   - **지역**: 전북 완주군
   - **작업 요약**: 기타: 고랑/잔재 정리
   - **저장 시각**: 2026-08-31 23:52:02

2. **작업일**: 2023-08-15
   - **작물**: 고추
   - **지역**: 전북 완주군
   - **작업 요약**: 관수: 관수, 시비: 웃거름
   - **저장 시각**: 2026-08-31 23:51:40

3. **작업일**: 2023-08-18
   - **작물**: 고추
   - **지역**: 전북 완주군
   - **작업 요약**: 방제: 방제
   - **저장 시각**: 2026-08-31 23:51:52


In [11]:
out_e = show_run(SAMPLE_QUESTIONS[4])


E. 토마토(자료 없음)
Q: 오늘 토마토 하우스 정식하고 점적관수 시작했어. 일지에 품종이랑 오늘 기온 넣어서 작성해줘.
호출 Tool: parse_work_memo
  → 메모 해석 {'memo': '오늘 토마토 하우스 정식하고 점적관수 시작했어.'}
  [결과:parse_work_memo] {"work_date": "", "region": "", "crop": "토마토", "variety": "", "work_items": [{"category": "정식", "description": "정식", "material": "", "amount": ""}, {"category": "관수", "description": "관수", "material": 

[최종 답]

토마토는 창고에 없는 작물이므로 일지에 저장할 수 없습니다. 기온과 품종을 포함한 정보를 작성할 수 없습니다. 다른 작물로 작업을 진행해 주세요.


## STEP 7. 결과 정리


In [12]:
runs = [
    ("A", SAMPLE_QUESTIONS[0], out_a),
    ("B", SAMPLE_QUESTIONS[1], out_b),
    ("C", SAMPLE_QUESTIONS[2], out_c),
    ("D", SAMPLE_QUESTIONS[3], out_d),
    ("E", SAMPLE_QUESTIONS[4], out_e),
]

saved = load_diary_table()


def diary_ids_for(crop, date):
    if saved.empty:
        return ""
    hit = saved[(saved["crop"].astype(str) == crop) & (saved["work_date"].astype(str) == date)]
    return ", ".join(hit["diary_id"].astype(str))


def weather_ok(answer, date, must_have):
    text = answer or ""
    return date in text or any(str(x) in text for x in must_have)


rows = []
for key, q, out in runs:
    tools = out["tools"]
    ans = out["answer"]
    pest_invented = any(
        word in ans for word in ["마이신", "스트로빌루린", "만코제브"]
    )
    if key == "B":
        pest_flag = "창작 의심" if pest_invented else "약제명 미기재 유지"
    else:
        pest_flag = "-"
    weather_flag = "-"
    did = ""
    note = ""
    if key == "A":
        weather_flag = "반교리 2023-08-15 인용" if ("2023-08-15" in ans or "24" in ans) else "확인 필요"
        did = diary_ids_for("고추", "2023-08-15")
        note = "열매 개수 단정 여부 확인"
    elif key == "B":
        weather_flag = "반교리 2023-08-18" if "2023-08-18" in ans or "20" in ans else "확인 필요"
        did = diary_ids_for("고추", "2023-08-18")
        note = "search_pest_info 사용"
    elif key == "C":
        weather_flag = "반교리 173mm" if "173" in ans else "확인 필요"
        if "2458" in ans:
            weather_flag += " / 이서면 이상치 인용됨(주의)"
        did = diary_ids_for("고추", "2023-08-10")
        note = "수확량 창작 금지"
    elif key == "D":
        note = "새 일지 생성 없이 목록 조회"
        did = "목록 조회"
    elif key == "E":
        replaced = "고추" in ans and "토마토" not in ans
        note = "고추로 대체함(감점)" if replaced else "토마토 자료 없음 명시"
        weather_flag = "기온 창작 없이 부족 명시"
    rows.append(
        {
            "시나리오": key,
            "기대한 Tool": ", ".join(q["expected"]) if isinstance(q["expected"], list) else q["expected"],
            "실제 호출 Tool": ", ".join(tools) if tools else "(없음)",
            "기상 근거": weather_flag,
            "약제 창작 여부": pest_flag,
            "저장 diary_id": did,
            "비고": note,
        }
    )

summary = pd.DataFrame(rows)
display(summary)


,시나리오,기대한 Tool,실제 호출 Tool,기상 근거,약제 창작 여부,저장 diary_id,비고
0,A,"parse_work_memo, get_diary_weather, get_crop_field_context, save_farm_diary","parse_work_memo, get_diary_weather, get_crop_field_context, save_farm_diary",반교리 2023-08-15 인용,-,1,열매 개수 단정 여부 확인
1,B,"parse_work_memo, get_diary_weather, search_pest_info, save_farm_diary","parse_work_memo, get_diary_weather, search_pest_info, get_crop_field_context...",반교리 2023-08-18,약제명 미기재 유지,2,search_pest_info 사용
2,C,"parse_work_memo, get_diary_weather, save_farm_diary","parse_work_memo, get_diary_weather, get_crop_field_context, save_farm_diary",반교리 173mm,-,3,수확량 창작 금지
3,D,query_saved_diaries,query_saved_diaries,-,-,목록 조회,새 일지 생성 없이 목록 조회
4,E,자료 없음 명시,parse_work_memo,기온 창작 없이 부족 명시,-,,토마토 자료 없음 명시


In [13]:
print("[farm_diary 테이블]")
saved = load_diary_table()
display(saved)

print("\n[narrative_ko 샘플]")
if not saved.empty:
    sample = saved.iloc[0]
    print(f"diary_id={sample['diary_id']} / {sample['work_date']} / {sample['crop']}")
    print(sample["narrative_ko"])
    print("\n[weather_json]")
    print(sample["weather_json"])
    print("\n[work_items_json]")
    print(sample["work_items_json"])
else:
    print("저장된 일지가 없습니다.")


[farm_diary 테이블]


,diary_id,work_date,region,crop,variety,weather_json,...,growth_note,pest_note,special_note,next_plan,narrative_ko,saved_at
0,1,2023-08-15,전북 완주군,고추,,"{""tmin_c"": 24.0, ""tmax_c"": 32.0, ""tavg_c"": 27.12, ""humidity_pct"": 85.83, ""ra...",...,열매는 달렸는데 갯수는 세지 못함.,,,,2023년 8월 15일 완주 고추밭에서 낮에 더워서 물을 한번 주고 웃거름을 조금 했습니다. 열매는 달렸으나 갯수는 세지 못했습니다.,2026-08-31 23:51:40
1,2,2023-08-18,전북 완주군,고추,,"{""tmin_c"": 24.0, ""tmax_c"": 32.0, ""tavg_c"": 27.33, ""humidity_pct"": 86.04, ""ra...",...,,"잎에 갈색 반점 증상 발생. 갈색점무늬병, 점무늬병 등 가능성 있음.",,약제명 미기재로 추가 방제 계획 필요,"2023년 8월 18일, 전북 완주군 고추에서 방제를 실시하였고, 잎에 갈색 반점이 발생하였습니다. 기온은 24도에서 32도 사이였으며,...",2026-08-31 23:51:52
2,3,2023-08-10,전북 완주군,고추,,"{""tmin_c"": 24.0, ""tmax_c"": 25.0, ""tavg_c"": 24.67, ""humidity_pct"": 93.12, ""ra...",...,2024년 참고 생육입니다. 2023 작업일 기상·당일 열매 수로 쓰지 마세요.,,,,비가 많이 와서 고추밭 물 빠지나 보고 고랑 정리만 했음. 수확은 안 함.,2026-08-31 23:52:02



[narrative_ko 샘플]
diary_id=1 / 2023-08-15 / 고추
2023년 8월 15일 완주 고추밭에서 낮에 더워서 물을 한번 주고 웃거름을 조금 했습니다. 열매는 달렸으나 갯수는 세지 못했습니다.

[weather_json]
{"tmin_c": 24.0, "tmax_c": 32.0, "tavg_c": 27.12, "humidity_pct": 85.83, "rainfall_mm": 0.0, "station": "완주군 반교리", "source_year": "2023", "missing_reason": ""}

[work_items_json]
[{"category": "관수", "description": "관수", "material": "", "amount": ""}, {"category": "시비", "description": "웃거름", "material": "", "amount": ""}]
